# 04 · Análise de qualidade dos dados

**Objetivo da etapa:** avaliar a qualidade de **cada atributo** do conjunto de dados, identificar
problemas e mostrar como eles foram tratados para não afetar as respostas às perguntas de negócio.

| Dimensão | Pergunta | Onde é verificada |
|---|---|---|
| Completude | Há valores ausentes? O que a ausência significa? | Seções 1 e 5 |
| Unicidade | As chaves são únicas? Há duplicatas na origem? | Seção 2 |
| Consistência | Há contradições entre atributos ou entre tabelas? | Seções 3 e 4 |
| Conformidade | Os valores respeitam o domínio declarado no Catálogo de Dados? | Seção 5 |
| Acurácia | Há valores implausíveis (fora da faixa esperada)? | Seção 5 |
| Atualidade | O período coberto é completo e atual? | Seção 6 |

A Seção 5 é **automática**: lê o `catalogo/catalogo.yaml` (domínios declarados *a priori*) e
confere cada atributo da camada Gold contra ele.

### Configuração do ambiente
Esta célula prepara o ambiente de execução. No Google Colab, ela:
1. clona o repositório do projeto, que contém o módulo `scripts/common.py`;
2. instala o PySpark e o Delta Lake;
3. monta o Google Drive, onde o lakehouse fica persistido.

Na primeira execução, o Colab pede autorização de acesso ao Drive.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/FilippeDumont/mvp-olist-atrasos-entrega.git"
REPO_DIR = "/content/repo"

if "google.colab" in sys.modules:
    if not os.path.isdir(f"{REPO_DIR}/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    # O Spark precisa de Java: instala um JDK apenas se a máquina não tiver um
    if subprocess.run(["bash", "-c", "command -v java"], capture_output=True).returncode != 0:
        subprocess.run(["bash", "-c", "apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pyspark==3.5.3", "delta-spark==3.2.1", "kagglehub"], check=True)
    sys.path.insert(0, f"{REPO_DIR}/scripts")
else:  # execução local de teste
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "scripts")))

import common as C

C.mount_drive()
spark = C.get_spark()
print("Spark", spark.version, "| Delta Lake habilitado")

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=42987 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=
Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=42987 -Dhttp.nonProxyHosts=localhost|127.0.0.1

26/11/09 17:26:14 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/11/09 17:26:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


26/11/09 17:26:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.3 | Delta Lake habilitado


In [2]:
import re, yaml
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType, BooleanType

C.banner("04 · Análise de qualidade dos dados")
pd.set_option("display.max_rows", 200, "display.width", 200, "display.max_colwidth", 60)

SILVER = ["pedidos", "clientes", "vendedores", "itens_pedido", "pagamentos", "avaliacoes",
          "produtos", "geolocalizacao_cep"]
GOLD = ["fato_pedido", "dim_cliente", "dim_vendedor", "dim_produto", "dim_tempo"]
C.register_views("bronze", ["orders", "order_reviews", "geolocation", "sellers", "customers"])
C.register_views("silver", SILVER)
C.register_views("gold", GOLD)
sql = lambda q: spark.sql(q).toPandas()

04 · Análise de qualidade dos dados
Ambiente: VM Linux em nuvem (fora do Colab) | run_id: 20261109T172612Z-15d2c5
Persistência (disco persistente da VM): /home/claude/mvp_olist_lakehouse


## 1. Perfil por atributo (todas as tabelas da camada Silver)
Para cada atributo: tipo, completude (% de nulos), cardinalidade (valores distintos) e faixa
observada (mínimo e máximo; para texto, comprimento mínimo e máximo).

In [3]:
perfil = []
for t in SILVER:
    df = C.read_table("silver", t)
    n = df.count()
    for f in df.schema.fields:
        c, dt = f.name, f.dataType
        col = F.length(c) if isinstance(dt, StringType) else F.col(c).cast("int") if isinstance(dt, BooleanType) else F.col(c)
        r = df.agg(F.sum(F.col(c).isNull().cast("int")).alias("nulos"), F.countDistinct(c).alias("distintos"),
                   F.min(col).alias("min"), F.max(col).alias("max")).first()
        perfil.append({"tabela": t, "atributo": c, "tipo": dt.simpleString(), "linhas": n,
                       "nulos": int(r.nulos or 0), "pct_nulos": round(100 * (r.nulos or 0) / n, 2),
                       "distintos": r.distintos,
                       "min": ("len " if isinstance(dt, StringType) else "") + str(r["min"]),
                       "max": ("len " if isinstance(dt, StringType) else "") + str(r["max"])})
perfil = pd.DataFrame(perfil)
with C.Stage("04_qualidade", "gold.dq_perfil_atributos") as st:
    st.rows = C.write_table(spark.createDataFrame(perfil.astype(str)), "gold", "dq_perfil_atributos")
perfil

[SUCESSO] 04_qualidade -> gold.dq_perfil_atributos | 57 linhas | 4.56s


,tabela,atributo,tipo,linhas,nulos,pct_nulos,distintos,min,max
0,pedidos,pedido_id,string,99441,0,0.00,99441,len 32,len 32
1,pedidos,cliente_id,string,99441,0,0.00,99441,len 32,len 32
2,pedidos,status,string,99441,0,0.00,8,len 7,len 11
3,pedidos,dt_compra,timestamp,99441,0,0.00,98875,2016-09-04 21:15:19,2018-10-17 17:30:18
4,pedidos,dt_aprovacao,timestamp,99441,160,0.16,90733,2016-09-15 12:16:38,2018-09-03 17:40:06
5,pedidos,dt_postagem,timestamp,99441,1783,1.79,81018,2016-10-08 10:34:01,2018-09-11 19:48:28
6,pedidos,dt_entrega,timestamp,99441,2965,2.98,95664,2016-10-11 13:46:32,2018-10-17 13:22:46
7,pedidos,dt_entrega_prometida,date,99441,0,0.00,459,2016-09-30,2018-11-12
8,pedidos,dq_postagem_antes_compra,boolean,99441,0,0.00,2,0,1
9,pedidos,dq_entrega_antes_postagem,boolean,99441,0,0.00,2,0,1


**Leitura do perfil:**
- **Pedidos:** as datas ausentes (aprovação, postagem e entrega) correspondem a pedidos que não
  chegaram àquela fase, como cancelados ou em trânsito. Ou seja, são ausências *legítimas*.
- **Avaliações:** título e comentário são opcionais para o cliente, por isso têm muitos nulos.
- **Produtos:** os 610 produtos sem categoria são exatamente os mesmos que não têm nome, descrição
  nem quantidade de fotos. É um único grupo de cadastros incompletos na origem.
- **Itens:** o `dt_limite_postagem` vai até **2020**, uma data implausível para pedidos de 2016–2018.
  São só 4 itens, de 3 pedidos, sem efeito material sobre a regra G04 (postagem atrasada).
- **Vendedores:** só 23 das 27 UFs têm vendedores, o que é coerente com a concentração do comércio
  eletrônico no Sul e no Sudeste.

## 2. Unicidade: chaves e duplicatas

In [4]:
unicidade = sql("""
SELECT 'silver.pedidos'      AS tabela, 'pedido_id' AS chave, count(*) AS linhas, count(*) - count(DISTINCT pedido_id) AS duplicatas FROM silver_pedidos
UNION ALL SELECT 'silver.clientes', 'cliente_id', count(*), count(*) - count(DISTINCT cliente_id) FROM silver_clientes
UNION ALL SELECT 'silver.vendedores', 'vendedor_id', count(*), count(*) - count(DISTINCT vendedor_id) FROM silver_vendedores
UNION ALL SELECT 'silver.produtos', 'produto_id', count(*), count(*) - count(DISTINCT produto_id) FROM silver_produtos
UNION ALL SELECT 'silver.itens_pedido', 'pedido_id + item_seq', count(*), count(*) - count(DISTINCT pedido_id, item_seq) FROM silver_itens_pedido
UNION ALL SELECT 'silver.pagamentos', 'pedido_id + pagamento_seq', count(*), count(*) - count(DISTINCT pedido_id, pagamento_seq) FROM silver_pagamentos
UNION ALL SELECT 'silver.avaliacoes (após R06)', 'pedido_id', count(*), count(*) - count(DISTINCT pedido_id) FROM silver_avaliacoes
UNION ALL SELECT 'bronze.order_reviews (origem)', 'review_id', count(*), count(*) - count(DISTINCT review_id) FROM bronze_order_reviews
UNION ALL SELECT 'bronze.order_reviews (origem)', 'order_id', count(*), count(*) - count(DISTINCT order_id) FROM bronze_order_reviews
UNION ALL SELECT 'bronze.geolocation (origem)', 'linha inteira', count(*), count(*) - count(DISTINCT geolocation_zip_code_prefix, geolocation_lat, geolocation_lng, geolocation_city, geolocation_state) FROM bronze_geolocation
UNION ALL SELECT 'gold.fato_pedido', 'pedido_id', count(*), count(*) - count(DISTINCT pedido_id) FROM gold_fato_pedido
""")
unicidade

,tabela,chave,linhas,duplicatas
0,silver.pedidos,pedido_id,99441,0
1,silver.clientes,cliente_id,99441,0
2,silver.vendedores,vendedor_id,3095,0
3,silver.produtos,produto_id,32951,0
4,silver.itens_pedido,pedido_id + item_seq,112650,0
5,silver.pagamentos,pedido_id + pagamento_seq,103886,0
6,silver.avaliacoes (após R06),pedido_id,98673,0
7,bronze.order_reviews (origem),review_id,99224,814
8,bronze.order_reviews (origem),order_id,99224,551
9,bronze.geolocation (origem),linha inteira,1000163,261831


**Leitura:**
- Todas as chaves das camadas Silver e Gold são únicas.
- Na origem há dois problemas de unicidade:
  1. **avaliações:** o mesmo `review_id` aparece em pedidos diferentes, e alguns pedidos têm mais de
     uma avaliação. A regra R06 mantém uma avaliação por pedido, sem isso a nota seria contada em
     dobro na tabela fato;
  2. **geolocalização:** a base tem centenas de milhares de linhas idênticas. A regra R05 resolve
     isso ao agregar por prefixo de CEP.
- O `cliente_id` é único **por pedido**, não por pessoa. A pessoa é identificada por
  `cliente_unico_id`, como mostra a seção 1.

## 3. Consistência entre tabelas (integridade referencial)

In [5]:
integridade = sql("""
SELECT 'itens -> pedidos' AS relacao, count(*) AS orfaos FROM silver_itens_pedido i LEFT ANTI JOIN silver_pedidos p USING (pedido_id)
UNION ALL SELECT 'itens -> produtos', count(*) FROM silver_itens_pedido i LEFT ANTI JOIN silver_produtos p USING (produto_id)
UNION ALL SELECT 'itens -> vendedores', count(*) FROM silver_itens_pedido i LEFT ANTI JOIN silver_vendedores v USING (vendedor_id)
UNION ALL SELECT 'pedidos -> clientes', count(*) FROM silver_pedidos p LEFT ANTI JOIN silver_clientes c USING (cliente_id)
UNION ALL SELECT 'pagamentos -> pedidos', count(*) FROM silver_pagamentos g LEFT ANTI JOIN silver_pedidos p USING (pedido_id)
UNION ALL SELECT 'avaliacoes -> pedidos', count(*) FROM silver_avaliacoes a LEFT ANTI JOIN silver_pedidos p USING (pedido_id)
UNION ALL SELECT 'pedidos sem itens', count(*) FROM silver_pedidos p LEFT ANTI JOIN silver_itens_pedido i USING (pedido_id)
UNION ALL SELECT 'pedidos sem pagamento', count(*) FROM silver_pedidos p LEFT ANTI JOIN silver_pagamentos g USING (pedido_id)
UNION ALL SELECT 'pedidos sem avaliação', count(*) FROM silver_pedidos p LEFT ANTI JOIN silver_avaliacoes a USING (pedido_id)
UNION ALL SELECT 'CEP de cliente sem coordenada', count(*) FROM silver_clientes c LEFT ANTI JOIN silver_geolocalizacao_cep g USING (cep_prefixo)
UNION ALL SELECT 'CEP de vendedor sem coordenada', count(*) FROM silver_vendedores v LEFT ANTI JOIN silver_geolocalizacao_cep g USING (cep_prefixo)
""")
integridade

,relacao,orfaos
0,itens -> pedidos,0
1,itens -> produtos,0
2,itens -> vendedores,0
3,pedidos -> clientes,0
4,pagamentos -> pedidos,0
5,avaliacoes -> pedidos,0
6,pedidos sem itens,775
7,pedidos sem pagamento,1
8,pedidos sem avaliação,768
9,CEP de cliente sem coordenada,279


**Leitura:**
- Não há registros órfãos: toda chave estrangeira aponta para um registro existente.
- Os **pedidos sem itens** são cancelados ou indisponíveis. Na tabela fato eles apontam para o
  membro "desconhecido" (regra G02).
- Os **CEPs sem coordenada** fazem a `distancia_km` ficar nula para poucos pedidos. Esses pedidos
  saem apenas da análise por distância (P2).

## 4. Consistência temporal (regra R08)

In [6]:
temporal = sql("""
SELECT
  sum(CAST(dq_postagem_antes_compra AS INT))   AS postagem_antes_da_compra,
  sum(CAST(dq_entrega_antes_postagem AS INT))  AS entrega_antes_da_postagem,
  sum(CAST(dq_entregue_sem_data AS INT))       AS status_entregue_sem_data,
  sum(CAST(dq_data_entrega_sem_status AS INT)) AS data_entrega_sem_status_entregue,
  sum(CAST(dt_aprovacao < dt_compra AS INT))   AS aprovacao_antes_da_compra,
  sum(CAST(dt_entrega_prometida < to_date(dt_compra) AS INT)) AS promessa_antes_da_compra,
  sum(CAST(dq_inconsistente AS INT))           AS total_pedidos_sinalizados,
  round(100 * avg(CAST(dq_inconsistente AS INT)), 3) AS pct_sinalizados
FROM silver_pedidos""")
temporal.T.rename(columns={0: "pedidos"})

,pedidos
postagem_antes_da_compra,166.000
entrega_antes_da_postagem,23.000
status_entregue_sem_data,8.000
data_entrega_sem_status_entregue,6.000
aprovacao_antes_da_compra,0.000
promessa_antes_da_compra,0.000
total_pedidos_sinalizados,203.000
pct_sinalizados,0.204


**Leitura:** uma pequena fração dos pedidos tem datas contraditórias, sendo a principal delas
"postagem antes da compra", um erro de registro da origem. Como não existe fonte para corrigir
a data verdadeira, esses pedidos são **sinalizados** e excluídos das métricas de prazo
(regras G06 e `flag_entregue_valido`). O impacto nas taxas de atraso é desprezível, como se vê
pela fração de pedidos sinalizados.

## 5. Conformidade e acurácia contra o Catálogo de Dados (camada Gold, automática)

In [7]:
cat_path = os.path.join(C.REPO_ROOT, "catalogo", "catalogo.yaml")
catalogo = yaml.safe_load(open(cat_path, encoding="utf-8"))

def violacoes(df, nome, spec):
    """Devolve (nulos, fora_do_dominio, exemplos) de um atributo segundo o catálogo."""
    c, dom = F.col(nome), spec.get("dominio", {})
    nulos = df.filter(c.isNull()).count()
    if "categorias" in dom:
        validos = [str(v).lower() for v in dom["categorias"]]
        cond = ~F.lower(c.cast("string")).isin(validos)
    elif "regex" in dom:
        cond = ~c.cast("string").rlike(dom["regex"])
    elif "min" in dom:
        lo, hi = dom["min"], dom["max"]
        if spec["tipo"] == "data":
            cond = (c < F.lit(str(lo)).cast("date")) | (c > F.lit(str(hi)).cast("date"))
        else:
            cond = (c < lo) | (c > hi)
    else:
        return nulos, 0, ""
    fora = df.filter(c.isNotNull() & cond)
    exemplos = ", ".join(str(r[0]) for r in fora.select(c).distinct().limit(4).collect())
    return nulos, fora.count(), exemplos

linhas = []
for tabela, meta in catalogo["tabelas"].items():
    df = C.read_table("gold", tabela).cache()
    n = df.count()
    for atributo, spec in meta["atributos"].items():
        nulos, fora, ex = violacoes(df, atributo, spec)
        dom = spec.get("dominio", {})
        dom_txt = (f"[{dom['min']}, {dom['max']}]" if "min" in dom else
                   f"{len(dom['categorias'])} categorias" if "categorias" in dom else f"regex {dom.get('regex', '')}")
        problema_nulo = spec["obrigatorio"] and nulos > 0
        linhas.append({"tabela": tabela, "atributo": atributo, "tipo": spec["tipo"], "dominio": dom_txt,
                       "obrigatorio": spec["obrigatorio"], "nulos": nulos, "pct_nulos": round(100 * nulos / n, 2),
                       "fora_dominio": fora, "pct_fora": round(100 * fora / n, 3), "exemplos_fora": ex,
                       "status": "OK" if (fora == 0 and not problema_nulo) else "ATENÇÃO"})
    df.unpersist()

conformidade = pd.DataFrame(linhas)
with C.Stage("04_qualidade", "gold.dq_conformidade_catalogo") as st:
    st.rows = C.write_table(spark.createDataFrame(conformidade.astype(str)), "gold", "dq_conformidade_catalogo")
print(f"{len(conformidade)} atributos verificados | {(conformidade.status == 'OK').sum()} OK | "
      f"{(conformidade.status != 'OK').sum()} com atenção")
conformidade[conformidade.status != "OK"]

[SUCESSO] 04_qualidade -> gold.dq_conformidade_catalogo | 69 linhas | 3.46s
69 atributos verificados | 58 OK | 11 com atenção


,tabela,atributo,tipo,dominio,obrigatorio,nulos,pct_nulos,fora_dominio,pct_fora,exemplos_fora,status
12,fato_pedido,valor_pago,numérico,"[0.01, 15000]",False,1,0.00,3,0.003,0.0,ATENÇÃO
13,fato_pedido,tipo_pagamento,categórico,4 categorias,False,1,0.00,3,0.003,not_defined,ATENÇÃO
14,fato_pedido,parcelas,numérico,"[1, 24]",False,1,0.00,2,0.002,0,ATENÇÃO
15,fato_pedido,horas_aprovacao,numérico,"[0, 720]",False,363,0.37,7,0.007,"1450.8, 741.36, 4509.12, 738.48",ATENÇÃO
16,fato_pedido,dias_ate_postagem,numérico,"[0, 60]",False,1985,2.00,6,0.006,"62.81, 125.78, 66.05, 61.16",ATENÇÃO
17,fato_pedido,dias_transporte,numérico,"[0, 90]",False,3160,3.18,70,0.070,"136.05, 102.92, 171.16, 166.8",ATENÇÃO
18,fato_pedido,dias_entrega,numérico,"[0, 120]",False,3159,3.18,43,0.043,"126.01, 191.46, 194.05, 194.63",ATENÇÃO
19,fato_pedido,dias_prometidos,numérico,"[1, 90]",True,0,0.00,23,0.023,"107, 93, 141, 156",ATENÇÃO
20,fato_pedido,dias_atraso,numérico,"[-90, 120]",False,3159,3.18,31,0.031,"-147, 161, 138, 181",ATENÇÃO
33,dim_cliente,cidade,texto,regex ^[a-zà-ú' \-]+$,True,0,0.00,1,0.001,quilometro 14 do mutum,ATENÇÃO


**Atributos que exigem atenção e como foram tratados:**

| Atributo | Problema | Dimensão | Impacto nas perguntas | Tratamento |
|---|---|---|---|---|
| `dim_vendedor.cidade` | Cidade preenchida com UF junto (`sbc/sp`), com país (`..., brasil`), com um CEP (`04482255`) e até com um e-mail | Conformidade | Nenhum: as análises usam UF e região, não cidade | Mantidos e documentados; padronizar cidades é um trabalho futuro |
| `dim_cliente.cidade` | Um único caso com números (`quilometro 14 do mutum`, que é um nome real de localidade) | Conformidade | Nenhum | Falso positivo da regra; mantido |
| `fato_pedido.valor_pago` / `tipo_pagamento` | Pedidos com valor pago 0 (vouchers e `not_defined`); `not_defined` está fora das categorias válidas | Conformidade e acurácia | Nenhum: pagamento não entra nas perguntas | Mantido como está |
| `fato_pedido.parcelas` | Valor 0 de parcelas | Acurácia | Nenhum | Mantido como está |
| `fato_pedido.horas_aprovacao` | Poucas aprovações que levaram mais de 30 dias | Acurácia | Baixo: é um atributo descritivo | Mantido; aparece só na descrição |
| `fato_pedido.dias_entrega`, `dias_transporte`, `dias_ate_postagem`, `dias_atraso`, `dias_prometidos` | Valores além da faixa esperada (prazos muito longos) | Acurácia | **Relevante para P1–P5:** distorcem médias | As análises usam **medianas e percentuais** e a taxa de atraso (que é binária); os casos extremos são mantidos porque são atrasos reais |
| `fato_pedido.distancia_km`, `dim_*.lat/lng` | Nulos para CEPs sem coordenada | Completude | Baixo: só afeta P2 por distância | Esses pedidos ficam de fora apenas do recorte por distância |
| `fato_pedido.valor_pago` / `tipo_pagamento` / `parcelas` | 1 pedido sem pagamento registrado | Completude | Nenhum | Ausência documentada no catálogo |

## 6. Atualidade e cobertura temporal

In [8]:
cobertura = sql("""
SELECT t.ano_mes, count(*) AS pedidos,
       sum(CAST(f.status = 'delivered' AS INT)) AS entregues,
       round(100 * avg(CAST(f.status = 'delivered' AS INT)), 1) AS pct_entregues
FROM gold_fato_pedido f JOIN gold_dim_tempo t ON f.sk_data_compra = t.sk_data
GROUP BY t.ano_mes ORDER BY t.ano_mes""")
cobertura

,ano_mes,pedidos,entregues,pct_entregues
0,2016-09,4,1,25.0
1,2016-10,324,265,81.8
2,2016-12,1,1,100.0
3,2017-01,800,750,93.8
4,2017-02,1780,1653,92.9
5,2017-03,2682,2546,94.9
6,2017-04,2404,2303,95.8
7,2017-05,3700,3546,95.8
8,2017-06,3245,3135,96.6
9,2017-07,4026,3872,96.2


**Leitura e regra de recorte (A01):**
- Os meses das pontas têm volume residual: set–dez/2016, quando a operação estava começando, e
  set–out/2018, quando a extração aconteceu e muitos pedidos ainda não tinham sido entregues.
- Incluir pedidos ainda em trânsito **subestimaria o atraso**, porque só os que já chegaram entram
  na conta. Por isso, as análises consideram compras de **jan/2017 a ago/2018**, 20 meses completos.
- Quanto à **atualidade**, os dados são de 2016–2018. As conclusões descrevem o padrão operacional
  daquele período, e aplicá-las hoje exigiria dados recentes. Esse ponto volta na autoavaliação.

## 7. Síntese

In [9]:
sintese = pd.DataFrame([
    ("Completude", "Nulos legítimos (fases não atingidas, campos opcionais); 610 produtos sem categoria", "Documentado; categoria 'sem_categoria'"),
    ("Unicidade", f"{int(unicidade.loc[7,'duplicatas'])} review_id repetidos; {int(unicidade.loc[8,'duplicatas'])} avaliações extras por pedido; geolocalização com duplicatas", "R05, R06"),
    ("Consistência", f"{int(temporal.total_pedidos_sinalizados[0])} pedidos com datas contraditórias ({float(temporal.pct_sinalizados[0])}%)", "R08 sinaliza; G06 exclui das métricas de prazo"),
    ("Conformidade", f"{int((conformidade.status != 'OK').sum())} de {len(conformidade)} atributos com algum valor fora do domínio declarado", "Ver tabela da seção 5"),
    ("Acurácia", "Coordenadas fora do Brasil; peso ou dimensão 0; prazos extremos", "R05, R07; medianas e percentuais"),
    ("Atualidade", "Bordas do período incompletas; dados de 2016–2018", "A01: recorte jan/2017–ago/2018"),
], columns=["dimensão", "achado", "tratamento"])
sintese

,dimensão,achado,tratamento
0,Completude,"Nulos legítimos (fases não atingidas, campos opcionais);...",Documentado; categoria 'sem_categoria'
1,Unicidade,814 review_id repetidos; 551 avaliações extras por pedid...,"R05, R06"
2,Consistência,203 pedidos com datas contraditórias (0.204%),R08 sinaliza; G06 exclui das métricas de prazo
3,Conformidade,11 de 69 atributos com algum valor fora do domínio decla...,Ver tabela da seção 5
4,Acurácia,Coordenadas fora do Brasil; peso ou dimensão 0; prazos e...,"R05, R07; medianas e percentuais"
5,Atualidade,Bordas do período incompletas; dados de 2016–2018,A01: recorte jan/2017–ago/2018
